# 阶段 C：Session 与任务级服务证据浏览

这是只读分析入口，使用保存的真实摘要，不启动模型、HTTP、GPU或工具任务，不写入图表/结果。当前 Notebook 不内嵌任何性能数字。

默认等待 `summary/final-r1.json`。实际pilot完成后可以显式修改 `SUMMARY_RELATIVE` 并开启 `INCLUDE_NON_MEASUREMENT` 浏览，但pilot/Profile/warmup不能充当正式容量结果。固定回放没有任务成功率，动态闭环的调用数和历史可能不同。Session时延SLO未定义。


In [ ]:
from pathlib import Path
import hashlib
import json
import math

candidates = [Path.cwd(), *Path.cwd().parents]
WORKSPACE = next((p for p in candidates if (p / 'research/dsv4-flash0731/sessions').is_dir()), None)
if WORKSPACE is None:
    raise RuntimeError('Run from this workspace or a directory below it.')
PHASE = WORKSPACE / 'research/dsv4-flash0731/sessions'
SUMMARY_RELATIVE = 'summary/final-r1.json'  # Change only to an existing real output.
INCLUDE_NON_MEASUREMENT = False
VERIFY_ALL_INPUT_BYTES = False  # Optional potentially large read-only byte audit.
summary_path = PHASE / SUMMARY_RELATIVE
report = None
if summary_path.is_file():
    raw = summary_path.read_bytes()
    report = json.loads(raw)
    if report.get('status') != 'PASS' or report.get('errors'):
        raise ValueError('The selected summary has not passed evidence analysis.')
    print('Summary:', summary_path)
    print('SHA256:', hashlib.sha256(raw).hexdigest())
    print('Experiments:', len(report['experiments']), 'phases:', len(report['stages']))
else:
    print('NO REAL SUMMARY SELECTED:', summary_path)
    print('No performance values will be created or inferred.')


## 来源与完整性

摘要PASS表示它的输入一致性检查通过，不等于业务SLO、质量非劣或容量认证。下面可选择重新读取输入清单；若证据来自解包目录，绝对路径按项目相对位置映射，不修改原记录。默认只显示清单数量，避免重复读取大文件；这时本Notebook没有重新验证全部原始字节。


In [ ]:
def locate_original(path):
    original = Path(path)
    if original.is_file():
        return original
    marker = '/research/dsv4-flash0731/'
    if marker in str(original):
        candidate = WORKSPACE / 'research/dsv4-flash0731' / str(original).split(marker, 1)[1]
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(path)

if report:
    ledger = report['input_ledger']
    print('Recorded input files:', len(ledger))
    if VERIFY_ALL_INPUT_BYTES:
        for original, info in ledger.items():
            path = locate_original(original)
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != info['sha256'] or path.stat().st_size != info['bytes']:
                raise ValueError('Changed evidence: ' + str(path))
        print('READ-ONLY INPUT BYTE CHECK: PASS')
    else:
        print('Raw bytes not rechecked by this Notebook; use the independent audit/archive records.')


## 会话吞吐、时延与任务成本

分母是完整cohort wall（含drain、工具、评分及有界清理），不是仅GPU时间。成功任务数与协议完成会话数分开。每成功任务Token成本分摊整个cohort的全部尝试；缺少失败调用usage时只得到已知Token成本下界，不能把它宣称完整成本。纯回放的成功任务列保持空值。


In [ ]:
def fmt(value, digits=3):
    if value is None:
        return '—'
    if isinstance(value, float):
        return f'{value:.{digits}f}'
    return str(value)

stages = [] if report is None else [s for s in report['stages']
    if s.get('included_as_measurement') or INCLUDE_NON_MEASUREMENT]
columns = ['experiment', 'phase', 'role', 'kind', 'cache', 'N', 'task successes',
           'session/s', 'task/s', 'visible P95 s', 'cohort P95 s', 'output tok/s',
           'prompt/success', 'token cost complete', 'window classification']
print(' | '.join(columns))
for stage in stages:
    p, s = stage['phase'], stage['summary']
    cost = s.get('tokens_per_successful_task') or {}
    values = [stage['experiment'], p['id'], p['role'], p['kind'], p['cache_scope'],
              s['sessions'], s['successful_tasks'], s['completed_sessions_s'], s['successful_tasks_s'],
              s['user_visible_latency_s']['p95'], s['session_latency_s']['p95'], s['output_tokens_s'],
              cost.get('prompt'), cost.get('token_usage_complete'), stage['stability']['classification']]
    print(' | '.join(fmt(v) for v in values))
if not stages:
    print('No eligible real phases. No numerical comparison is available.')


## 全部实际执行与失败实验诊断

以下清单覆盖所有选定ID，包括warmup、pilot和整轮FAILED。正常测量接受集是冻结分析器的`included_as_measurement=True`子集；失败运行中实际执行的confirm仍有单独计数，不能解释为未执行。模型调用量也按全部实际请求统计。

失败实验下表只作诊断：即使某个phase已完成，也不升级整轮FAILED为健康运行，不纳入正常排名或容量。时延只覆盖协议完成样本，需同时看协议完成/错误数。未执行窗口从保存的原plan与实际phase列表对比，显示NOT EXECUTED，不填零性能。


In [ ]:
execution_inventory_rows = []
failed_run_rows = []
unstarted_phase_rows = []
execution_inventory_plan_inputs = []
if report:
    for experiment in report['experiments']:
        selected = [s for s in report['stages'] if s['experiment'] == experiment['id']]
        accepted = [s for s in selected if s['included_as_measurement']]
        failed_confirms = [s for s in selected if not experiment['terminal_success'] and s['phase']['role'] == 'confirm']
        item = dict(experiment=experiment['id'], execution_status=experiment['execution_status'],
            terminal_success=experiment['terminal_success'], supervisor_exit_code=experiment['supervisor_exit_code'],
            all_executed_phases=len(selected), all_executed_sessions=sum(s['summary']['sessions'] for s in selected),
            all_executed_agent_attempts=sum(s['summary']['sessions'] for s in selected if s['phase']['kind'] == 'agent'),
            all_executed_model_calls=sum(s['summary']['model_calls'] for s in selected),
            normal_accepted_sessions=sum(s['summary']['sessions'] for s in accepted),
            normal_accepted_agent_attempts=sum(s['summary']['sessions'] for s in accepted if s['phase']['kind'] == 'agent'),
            failed_run_executed_confirm_sessions=sum(s['summary']['sessions'] for s in failed_confirms),
            failed_run_executed_confirm_model_calls=sum(s['summary']['model_calls'] for s in failed_confirms))
        execution_inventory_rows.append(item)
        plan_path = PHASE.parent / 'scenarios/results' / experiment['id'] / 'session-plan.json'
        plan_bytes = plan_path.read_bytes()
        expected = [info for original, info in report['input_ledger'].items()
                    if original.endswith('/' + experiment['id'] + '/session-plan.json')]
        assert len(expected) == 1 and expected[0] == dict(sha256=hashlib.sha256(plan_bytes).hexdigest(), bytes=len(plan_bytes))
        execution_inventory_plan_inputs.append(str(plan_path))
        plan = json.loads(plan_bytes)
        executed_ids = {s['phase']['id'] for s in selected}
        for phase in plan['phases']:
            if phase['id'] not in executed_ids:
                unstarted_phase_rows.append(dict(experiment=experiment['id'], phase=phase['id'],
                    kind=phase['kind'], role=phase['role'], rate=phase.get('rate'), seed=phase.get('seed'), status='NOT EXECUTED'))
        if not experiment['terminal_success']:
            assert not accepted, 'A failed run cannot enter normal accepted measurements.'
            for stage in selected:
                stats = stage['summary']
                failed_run_rows.append(dict(experiment=experiment['id'], phase=stage['phase']['id'],
                    role=stage['phase']['role'], kind=stage['phase']['kind'], rate=stage['phase'].get('rate'), seed=stage['phase'].get('seed'),
                    sessions=stats['sessions'], protocol_completed=stats['protocol_completed'], strict_successes=stats['successful_tasks'],
                    model_calls=stats['model_calls'], terminal_reasons=stats['terminal_reasons'],
                    visible_completed_p50_p95_p99_s={key:stats['user_visible_latency_s'][key] for key in ('p50','p95','p99')},
                    all_attempt_token_cost=stats.get('tokens_per_successful_task'),
                    classification=stage['stability']['classification'], blockers=stage['stability'].get('reasons'),
                    normal_ranking_or_capacity_eligible=False, exclusion=stage['exclusion_reason']))
    print('All executed experiment inventory (not a pooled performance estimate):')
    for item in execution_inventory_rows:
        print(item)
    fields = ('all_executed_phases', 'all_executed_sessions', 'all_executed_agent_attempts', 'all_executed_model_calls',
              'normal_accepted_sessions', 'normal_accepted_agent_attempts',
              'failed_run_executed_confirm_sessions', 'failed_run_executed_confirm_model_calls')
    print('Separate execution totals:', {field:sum(row[field] for row in execution_inventory_rows) for field in fields})
    print('Failed-run diagnostic rows: descriptive only, never normal ranking or healthy capacity:')
    for item in failed_run_rows:
        print(item)
    print('Saved-plan windows not executed within their original experiment:')
    for item in unstarted_phase_rows:
        print(item)
    if not unstarted_phase_rows:
        print('None within the selected experiments.')
else:
    print('No actual summary selected; no execution or failure values inferred.')


## 匹配条件、缓存与每轮变化

实际回放身份包括token IDs、输出预算、等待和到达计划，排除phase名称与cache/routing处理字段；处理因素仍由每条实际请求单独核验。只有回放身份相同才可把处理差异作为系统对照。动态Agent身份只匹配任务分配/预算，不能保证相同请求数量、长度或wire消息。

cached usage缺失与零命中不同；只有所有调用都提供该字段才显示整体fraction。命中计数是实际观测，cache标签不等于命中保证。


In [ ]:
if report:
    groups = {}
    for stage in stages:
        groups.setdefault(stage['arrival_workload_sha256'], []).append(stage)
    for identity, group in groups.items():
        print('\nWorkload identity:', identity)
        print(group[0].get('arrival_workload_identity', {}).get('scope', 'See protocol'))
        for stage in group:
            s = stage['summary']
            print(stage['experiment'], stage['phase']['id'], stage.get('treatment'),
                  'cache fraction:', fmt(s['cache_fraction']),
                  'known cache calls:', s['cached_usage_calls'], '/', s['successful_model_calls'])
            for turn, data in stage['by_turn'].items():
                print('  turn', turn, 'N', data['calls'], 'prompt P50', fmt(data['prompt_tokens']['p50']),
                      'cached P50', fmt(data['cached_tokens']['p50']),
                      'first-output P95 s', fmt(data['first_output_s']['p95']))


## 正式数据可视化

只有实际符合筛选条件的数据才绘图；不生成随机值或示例性能点。每个点是一个完整窗口，不是置信区间。不同类型、缓存、任务/shape与到达条件不能通过一张图合成全局冠军。


In [ ]:
import importlib.util
plot_source = PHASE / 'scripts/plot_session_comparisons.py'
plot_spec = importlib.util.spec_from_file_location('phasec_readonly_views', plot_source)
view_tools = importlib.util.module_from_spec(plot_spec)
plot_spec.loader.exec_module(view_tools)
comparison_views = []
if report:
    formal_rows, excluded_rows = view_tools.collect(report)
    comparison_views = view_tools.groups(formal_rows)
    print('Formal matched groups:', len(comparison_views), '; excluded phases:', len(excluded_rows))
    for view in comparison_views:
        print(view['id'], 'complete condition/seed grid:', view['complete_paired_grid'],
              'missing:', view['missing_pairs'])
        for row in view['rows']:
            print(' ', row['experiment'], row['phase_id'], 'N=', row['sessions'],
                  'visible-latency N=', row['visible_latency_samples'],
                  'workload SHA=', row['arrival_workload_sha256'])
if comparison_views:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('Matplotlib unavailable; exact comparison rows remain visible above.')
    else:
        for view in comparison_views:
            fig, axes = plt.subplots(1, 3, figsize=(max(13, len(view['conditions'])*2.5), 4.7))
            for ax, metric in zip(axes, view['metrics']):
                percent = metric in ('cache_fraction', 'task_success_rate')
                for si, seed in enumerate(view['seeds']):
                    selected = [r for r in view['rows'] if r['seed'] == seed and r[metric] is not None]
                    x = [view['conditions'].index(r['condition']) + (si-(len(view['seeds'])-1)/2)*.12 for r in selected]
                    values = [r[metric]*(100 if percent else 1) for r in selected]
                    ax.scatter(x, values, label=f'Seed {seed}', s=55)
                ax.set_xticks(range(len(view['conditions'])), [c.replace(' + ', '\n+ ') for c in view['conditions']])
                ax.set_ylabel(view_tools.LABELS[metric])
                ax.grid(axis='y', alpha=.2)
                ax.tick_params(axis='x', labelrotation=0, labelsize=9)
                if percent:
                    ax.set_ylim(-5, 105)
            axes[0].legend(fontsize=8)
            fig.suptitle(view['title'] + '\nOne point per actual cohort; no Session SLO or open-capacity claim.')
            fig.tight_layout()
            plt.show()
else:
    print('No real matched normal-performance groups: formal plots intentionally omitted.')


## 按具体任务的质量与成本、采样资源

同一任务反复执行不会增加任务种类。每task成功率分母为全部分配尝试；用户可见P95只覆盖自然完成样本，必须同时看完成数和失败类别。GPU能耗只能在整个phase平均分摊，不按token比例伪造单session归属。C1/C8、C1/C6不同seed/请求数分开展示，不将其比值解释为严格同工作量加速比。


In [ ]:
for stage in stages:
    p, total = stage['phase'], stage['summary']
    print('\n', stage['experiment'], p['id'], 'C=', p.get('concurrency'),
          'role=', p['role'], 'normal=', stage['included_as_measurement'])
    print('Source workload SHA:', stage['arrival_workload_sha256'])
    print('GPU evidence status:', stage['gpu']['status'],
          'sampled board J:', fmt(stage['gpu'].get('energy_j')),
          'J / successful task:', fmt(total.get('gpu_j_per_successful_task')))
    cost = total.get('tokens_per_successful_task')
    if cost:
        print('All-attempt known cost / success:', cost)
    for name, value in stage['by_workload'].items():
        print(' ', name, 'N=', value['sessions'], 'protocol complete=', value['protocol_completed'],
              'task successes=', value['successful_tasks'],
              'visible latency N=', value['user_visible_latency_s']['n'],
              'visible P95 s=', fmt(value['user_visible_latency_s']['p95']),
              'calls=', value['model_calls'], 'tools=', value['tool_calls'],
              'terminal reasons=', value['terminal_reasons'])


## 实际工作量与严格评分参考

独立工作量描述器读取raw Session JSONL。下面可选加载其实际产物，列出各任务输入/输出/cache分位数、所有尝试分摊的成功成本和墙钟分解。动态Agent长度与固定回放不同，约8K以内的实际任务不等于16K–64K真实任务质量验证。

严格任务成功与独立JSON内容参考分开；后者只对特定结构化任务且满足适用条件的回答评分，不回写原始分数，不自动算成性能成功任务。


In [ ]:
WORKLOAD_RELATIVE = 'summary/workloads-final-r1.json'  # Existing actual output only.
workload_path = PHASE / WORKLOAD_RELATIVE
if workload_path.is_file():
    workload_report = json.loads(workload_path.read_text())
    assert workload_report.get('status') == 'PASS'
    print('Workload descriptor:', workload_path, 'SHA256:', hashlib.sha256(workload_path.read_bytes()).hexdigest())
    selected = {(s['experiment'], s['phase']['id']) for s in stages}
    for item in workload_report['stages']:
        if (item['experiment'], item['phase']['id']) not in selected:
            continue
        print('\n', item['experiment'], item['phase']['id'], 'kind=', item['phase']['kind'], 'role=', item['phase']['role'])
        for task, data in [('ALL', item['summary']), *item['by_task'].items()]:
            lengths = {name: {key: value[key] for key in ('n', 'missing', 'p50', 'p95', 'p99', 'max')}
                       for name, value in data['request_lengths'].items()}
            print(task, 'attempts=', data['sessions'], 'strict successes=', data['successful_tasks'],
                  'calls=', data['model_calls'], 'actual tools=', data['actual_tool_calls'],
                  'synthetic waits=', data['synthetic_waits'])
            print('  actual per-call lengths:', lengths)
            print('  all-attempt token cost:', data['all_attempt_cost'])
        timing = item['summary']['timing']
        print('Phase cohort wall (s):', item['summary']['cohort_wall_s'])
        print('OVERLAPPING Session component sums (not cohort wall or CPU time):', timing['summed_session_components_s'])
        print('Complete decompositions/admitted:', timing['complete_decompositions'], timing['admitted_sessions'])
else:
    print('No real workload descriptor selected:', workload_path)

from collections import Counter
quality_rows = []
for experiment in ([] if report is None else report['experiments']):
    reference_path = PHASE / 'logs' / (experiment['id'] + '-json-reference-r1.json')
    if not reference_path.is_file():
        print('No independent JSON reference selected for', experiment['id'])
        continue
    reference = json.loads(reference_path.read_text())
    assert reference.get('status') == 'PASS' and reference.get('primary_unchanged') is True
    assert reference.get('applied_to_runtime') is False and reference.get('feedback_to_model') is False
    print('JSON reference:', reference_path, 'SHA256:', hashlib.sha256(reference_path.read_bytes()).hexdigest())
    for stage in stages:
        if stage['experiment'] != experiment['id'] or stage['phase']['kind'] != 'agent':
            continue
        answers = [s for s in reference['sessions'] if s['phase'] == stage['phase']['id']]
        assert len(answers) == stage['summary']['sessions']
        strict = sum(s['original_primary_success'] is True for s in answers)
        assert strict == stage['summary']['successful_tasks']
        eligible = [s for s in answers if s['reference_eligible']]
        print(' ', stage['phase']['id'], 'STRICT:', strict, '/', len(answers),
              '; separate JSON-reference passes/eligible:',
              sum(s['reference_passed'] is True for s in eligible), '/', len(eligible))
        regrade_path = PHASE / 'logs' / (experiment['id'] + '-objective-regrade-r1.json')
        regrade_rows = {}
        if regrade_path.is_file():
            regrade = json.loads(regrade_path.read_text())
            assert regrade.get('status') == 'PASS'
            regrade_rows = {(r['phase_id'], r['session_index']): r for r in regrade['sessions']}
            print('  Objective regrade:', regrade_path, 'SHA256:', hashlib.sha256(regrade_path.read_bytes()).hexdigest())
        for answer in answers:
            objective = regrade_rows.get((stage['phase']['id'], answer['session_index']))
            if objective:
                assert objective['recorded_task_success'] is answer['original_primary_success']
            checks = (answer.get('reference_grade') or {}).get('checks', {})
            quality_rows.append(dict(experiment=experiment['id'], phase=stage['phase']['id'],
                task=answer['task_id'], task_type=answer['kind'], strict_success=answer['original_primary_success'] is True,
                reference_eligible=answer['reference_eligible'], reference_passed=answer['reference_passed'],
                citation_gap=any(v.get('evidence_supported') is False for v in checks.values()),
                reference_error=(answer.get('reference_error') or {}).get('message'),
                primary_classification=objective['classification'] if objective else 'OBJECTIVE_RECHECK_NOT_SELECTED'))
print('\nQuality by experiment / phase / type and task; STRICT and JSON reference are separate:')
for experiment, phase in sorted({(r['experiment'], r['phase']) for r in quality_rows}):
    chosen = [r for r in quality_rows if (r['experiment'], r['phase']) == (experiment, phase)]
    for field in ('task_type', 'task'):
        for name in sorted({r[field] for r in chosen}):
            part = [r for r in chosen if r[field] == name]
            eligible = [r for r in part if r['reference_eligible']]
            print(experiment, phase, field, name,
                  'STRICT=', sum(r['strict_success'] for r in part), '/', len(part),
                  'separate JSON ref=', sum(r['reference_passed'] is True for r in eligible), '/', len(eligible),
                  'reference citation-gap cases=', sum(r['citation_gap'] for r in eligible),
                  'reference extraction/errors=', dict(Counter(r['reference_error'] for r in eligible if r['reference_error'])),
                  'primary outcomes=', dict(Counter(r['primary_classification'] for r in part)))
print('These are authored local tasks; repeating six tasks does not increase task diversity. Reference scores never alter main Goodput.')


## 开放式 Poisson Session 到达

固定并发与开放到达分开展示。按实际workload、缓存/路由、时长、角色、全局超时/保护界限和客户端身份分组，逐rate/seed核对精确工作量。每点是一个真实完成窗口；SCREEN_ONLY与单窗稳定均不自动成为重复稳定容量。全部已开始confirm的结论应看下面独立聚合审计。


In [ ]:
open_source = PHASE / 'scripts/plot_open_sessions.py'
open_spec = importlib.util.spec_from_file_location('phasec_open_views', open_source)
open_tools = importlib.util.module_from_spec(open_spec)
open_spec.loader.exec_module(open_tools)
open_views = []
if report:
    open_rows, open_excluded, open_plan_ledger = open_tools.collect(report)
    open_views = open_tools.groups(open_rows)
    print('Real eligible Poisson windows:', len(open_rows), '; excluded failed/non-measurement open windows:', open_excluded)
    for view in open_views:
        print('Group:', view['id'], 'contract:', view['contract'])
        print('Missing condition/rate/seed pairs:', view['missing_pairs'])
        for row in view['rows']:
            print(row['experiment'], row['phase_id'], 'rate=', row['rate'], 'seed=', row['seed'],
                  'N=', row['sessions'], 'strict wins=', row['successful_tasks'],
                  'protocol session/s=', row['protocol_sessions_s'], 'successful task/s=', row['successful_tasks_s'],
                  'visible P95=', row['visible_p95_s'], 'visible N=', row['visible_latency_n'],
                  'window class=', row['classification'], 'descriptive trend=', row['trend_verdict'], 'reasons=', row['stability_reasons'])
if open_views:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('Matplotlib unavailable; real open-window tables remain visible.')
    else:
        for view in open_views:
            open_tools.build_figure(view)
            plt.show()
else:
    print('No real eligible Poisson windows; no open performance plot generated.')


## 重复稳定确认的独立证据

只有配置、runtime、任务、缓存与路由、rate和全局服务界限一致，至少两种seed且全部已开始confirm窗口通过独立有限窗口规则，才能确认该点的有限协议稳定。下面不从图上的单窗标记重新推断结论，不认证Session业务SLO或精确最大容量。


In [ ]:
CONFIRMATION_RELATIVE = None  # Set only to an existing independent terminal audit.
if CONFIRMATION_RELATIVE:
    confirmation_path = PHASE / CONFIRMATION_RELATIVE
    confirmation = json.loads(confirmation_path.read_text())
    assert confirmation.get('status') == 'PASS'
    print('Independent confirmation audit:', confirmation_path)
    print('SHA256:', hashlib.sha256(confirmation_path.read_bytes()).hexdigest())
    print('Started windows:', confirmation['started_confirmation_windows'])
    print('Confirmed protocol points in full scan:', confirmation['confirmed_protocol_points'])
    print('Unresolved started identities blocking the full scan:', confirmation.get('unresolved_started_identity', []))
    for group in confirmation['groups']:
        identity = group['identity']
        print('Cohort:', group['cohort_sha256'], 'workload:', identity['workload'])
        print('Confirmation:', group['confirmation'])
        for window in group['windows']:
            print(' ', window['experiment'], window['phase_id'], 'seed=', window['seed'],
                  'classification=', window['classification'], 'tasks=', window['task_successes'],
                  'assigned N=', window['actual_sessions'])
else:
    print('No independent confirmation audit selected; no repeated-rate certification is inferred.')


## 按同条件累计的任务成本与逐轮分布

可选加载实际累计证据快照。源实验、全部已执行phase和正式计数必须与本Notebook选择一致；pilot/失败完整保留在清单，默认不进入正式成本表。仅同实验、同完整处理条件去ID/seed后汇总，role与duration不合并。

逐轮分布保留0基raw turn编号（例如6表示第7轮），条件是实际到达该轮；同时显示该轮Session数与全部分配尝试数。后续轮次样本较少不能解释为全体任务上下文变短。成本使用全部尝试，零成功为undefined、缺usage为已知下界。质量参考不替换严格主分，跨条件清单总数不构成全局质量/吞吐排名。


In [ ]:
CUMULATIVE_RELATIVE = None  # Select only an existing cumulative output for this exact summary.
cumulative_report = None
cumulative_counts_match = None
if CUMULATIVE_RELATIVE is not None:
    cumulative_path = PHASE / CUMULATIVE_RELATIVE
    cumulative_report = json.loads(cumulative_path.read_text())
    assert cumulative_report.get('status') == 'PASS' and cumulative_report.get('inputs_unchanged') is True
    cumulative_source = PHASE / 'scripts/summarize_session_workloads.py'
    assert cumulative_report.get('script_sha256') == hashlib.sha256(cumulative_source.read_bytes()).hexdigest(), 'Select a snapshot built by the current reviewed cumulative implementation.'
    assert report is not None, 'A cumulative snapshot requires its corresponding actual summary.'
    expected_ids = {e['id'] for e in report['experiments']}
    assert {e['experiment'] for e in cumulative_report['experiments']} == expected_ids
    expected_phases = {(s['experiment'], s['phase']['id']) for s in report['stages']}
    actual_phases = {(e['experiment'], phase) for e in cumulative_report['experiments'] for phase in e['phase_ids']}
    assert actual_phases == expected_phases, 'Cumulative snapshot omits or adds selected phases.'
    formal = [s for s in report['stages'] if s['included_as_measurement']]
    agent = [s for s in formal if s['phase']['kind'] == 'agent']
    expected_counts = dict(experiments=len(expected_ids), phases=len(expected_phases),
        all_sessions=sum(s['summary']['sessions'] for s in report['stages']),
        formal_sessions=sum(s['summary']['sessions'] for s in formal),
        formal_agent_attempts=sum(s['summary']['sessions'] for s in agent),
        formal_agent_strict_successes=sum(s['summary']['successful_tasks'] for s in agent),
        formal_agent_model_calls=sum(s['summary']['model_calls'] for s in agent))
    cumulative_counts_match = all(cumulative_report['inventory'][key] == value for key, value in expected_counts.items())
    assert cumulative_counts_match, 'Cumulative inventory contradicts the selected summary.'
    cumulative_bound_files = []
    def cumulative_portable_key(path):
        marker = '/research/dsv4-flash0731/'
        value = str(Path(path).resolve())
        return marker + value.split(marker, 1)[1] if marker in value else value
    def bind_cumulative_input(path):
        matches = [info for original, info in cumulative_report['input_ledger'].items()
                   if cumulative_portable_key(original) == cumulative_portable_key(path)]
        assert len(matches) == 1, 'Consumed evidence is absent or ambiguous in cumulative ledger: ' + str(path)
        raw = path.read_bytes()
        assert matches[0] == dict(sha256=hashlib.sha256(raw).hexdigest(), bytes=len(raw)), 'Consumed evidence changed since cumulative analysis: ' + str(path)
        cumulative_bound_files.append(str(path))
    for selected_input in (summary_path, cumulative_source, PHASE / 'scripts/describe_agent_workloads.py'):
        bind_cumulative_input(selected_input)
    for selection in cumulative_report['reference_inventory']:
        try:
            selected_reference = locate_original(selection['path'])
        except FileNotFoundError:
            selected_reference = None
        assert (selected_reference is not None) is selection['available'], 'Reference selection/availability changed since cumulative analysis.'
        if selected_reference is not None:
            bind_cumulative_input(selected_reference)
        print('Cumulative reference selection:', selection)
    print('Cumulative consumed-input bindings verified:', len(cumulative_bound_files))
    print('Cumulative snapshot:', cumulative_path, 'SHA256:', hashlib.sha256(cumulative_path.read_bytes()).hexdigest())
    print('Inventory only, not pooled performance:', cumulative_report['inventory'])
    for e in cumulative_report['experiments']:
        print('Experiment inventory:', e)
    for cohort in cumulative_report['cohorts']:
        if not cohort['included_as_measurement'] and not INCLUDE_NON_MEASUREMENT:
            continue
        print('\nCohort:', cohort['experiment'], 'formal=', cohort['included_as_measurement'],
              'treatment=', cohort['treatment'], 'phases=', cohort['source_phases'])
        print('Measured Session concurrency by source phase:', cohort['source_phase_concurrency'])
        for task, stats in [('ALL', cohort['summary']), *cohort['by_task'].items()]:
            print(task, 'attempts/strict=', stats['sessions'], '/', stats['successful_tasks'],
                  'all-attempt cost=', stats['all_attempt_cost'], 'separate quality reference=', stats['quality'])
        for turn in cohort['summary']['turn_distributions']:
            print('raw turn (zero-based; 6 = seventh round)=', turn['raw_turn_index'], 'reaching/assigned=', turn['sessions_reaching_turn'], '/', turn['assigned_sessions'],
                  'calls=', turn['model_calls'], 'failed=', turn['failed_model_calls'],
                  'usage known/missing=', turn['usage_known_calls'], '/', turn['usage_missing_calls'],
                  'request lengths=', turn['request_lengths'])
else:
    print('No cumulative snapshot selected; no cross-window cost or turn-distribution values are inferred.')


## 未认证事项

`STABLE_IN_WINDOW`只是一个窗口；有限稳定确认还需要同配置/同负载/同rate至少两个不同seed、全部已开始confirm窗口均通过。主分析器只输出单窗口分类；多窗口结论以独立 audit_session_confirmations.py 实际终态审计为准。Session延迟SLO未定义，所以本Notebook不输出SLO容量。GPU能耗是四卡采样板卡能耗，不是整机能耗或单session直接归因。

需同时阅读 `finite-session-v1.md`、`task-package.md`、独立终态审计以及最终实验登记。Notebook代码执行通过不代表实际模型、质量或吞吐已验证。
